# Load Gap Data from CSV

This section loads gap properties for multiple disks from the CSV file `gaps.csv` using pandas. The data includes disk name, gap radius, width, depth, and sigma values for each gap, enabling further analysis and extraction of gap-specific properties.

In [2]:
import numpy as np
import os

In [3]:
# with pickle
import sys
sys.path.append('D:/CPD_MPIA/Median_SNR/')  # or wherever disk_residuals_median_SNR.py is located

import pickle
with open('D:/CPD_MPIA/Median_SNR/disk_object_files/all_disks.pkl', 'rb') as f:
    all_disks = pickle.load(f)


In [4]:
import pandas as pd

# Path to the gap data CSV file
gap_csv_path = r'd:\CPD_MPIA\Median_SNR\disk_object_files\gaps.csv'

# Load the gap properties into a DataFrame
gap_df = pd.read_csv(gap_csv_path)

# Display the first few rows to verify
print(gap_df.head())

  disk_name  radius_au  radius_arcsec  width_au  width_arcsec  gap_depth  \
0    AA_Tau       11.0          0.082      28.1         0.209       0.01   
1    AA_Tau       64.3          0.478       8.2         0.061       0.44   
2    AA_Tau       79.8          0.593      10.2         0.076       0.34   
3    AA_Tau      105.3          0.782       4.9         0.036       0.94   
4    DM_Tau       13.5          0.094      12.7         0.088       0.08   

   r_in_au  r_in_arcsec  r_out_au  r_out_arcsec  
0      4.9        0.037      33.0         0.245  
1     60.3        0.448      68.5         0.508  
2     75.3        0.559      85.5         0.635  
3    103.1        0.766     108.0         0.802  
4      7.7        0.053      20.4         0.142  


# Define get_sigma_at_r Function

This function retrieves the sigma value for a given disk and gap radius from the loaded DataFrame. It matches both the disk name and the gap radius (in AU) and returns the corresponding sigma value. If no match is found, it returns None.

In [5]:
def get_sigma_at_radius(self, radius_au, robust_val="2.0", use_full_fov=True):
        """
        Returns the standard deviation (RMS noise) at a given radius (AU) for this disk in μJy.
        """

        radius_arcsec = radius_au / self.distance_pc
        suffix = "_FullFOV" if use_full_fov else ""
        sigma_file = os.path.join(
            r"D:\CPD_MPIA\Median_SNR",  # Add the base directory
            "Disk_Residual_Profile_Median_SNR", self.name,
            f"{self.name}_residual_radial_profile{suffix}_robust{robust_val}.txt"
        )

        prof = np.genfromtxt(sigma_file, comments="#")
        radii_arcsec = prof[:, 0] # first column: radius in arcsec
        sigma_prof_jybeam = prof[:, 2]   # [Jy/beam], second column: sigma in Jy/beam

        idx = np.nanargmin(np.abs(radii_arcsec - radius_arcsec)) # closest to requested radius
        sigma_beam_Jy = sigma_prof_jybeam[idx] # in Jy/beam 
        sigma_beam_uJy = sigma_beam_Jy * 1e6  # convert to μJy/beam

        return sigma_beam_uJy

# Test get_sigma_at_r with Example Disk and Radius

Test the function by extracting the sigma value for the disk 'AA_Tau' at a gap radius of 79.8 AU. This verifies that the function correctly locates and returns the expected value from the DataFrame.

In [6]:
for disk_name, radius_au in gap_df[['disk_name','radius_au']].values:
    disk_obj = all_disks.get(disk_name)
    if disk_obj is not None:
        sigma = disk_obj.get_sigma_at_radius(radius_au=radius_au, robust_val="2.0", use_full_fov=False)
        print(f"{disk_name} at {radius_au} AU: sigma = {sigma:.2f} μJy/beam")

AA_Tau at 11.0 AU: sigma = 72.51 μJy/beam
AA_Tau at 64.3 AU: sigma = 71.62 μJy/beam
AA_Tau at 79.8 AU: sigma = 136.85 μJy/beam
AA_Tau at 105.3 AU: sigma = 90.19 μJy/beam
DM_Tau at 13.5 AU: sigma = 340.87 μJy/beam
DM_Tau at 71.8 AU: sigma = 60.58 μJy/beam
DM_Tau at 102.5 AU: sigma = 39.18 μJy/beam
HD_135344B at 13.2 AU: sigma = 526.82 μJy/beam
HD_135344B at 66.5 AU: sigma = 10429.01 μJy/beam
HD_143006 at 21.8 AU: sigma = 506.74 μJy/beam
HD_143006 at 52.2 AU: sigma = 1331.93 μJy/beam
HD_34282 at 21.8 AU: sigma = 112.13 μJy/beam
HD_34282 at 59.3 AU: sigma = 187.32 μJy/beam
HD_34282 at 145.2 AU: sigma = 864.99 μJy/beam
HD_34282 at 188.2 AU: sigma = 138.94 μJy/beam
J1615 at 12.3 AU: sigma = 104.15 μJy/beam
J1615 at 82.6 AU: sigma = 128.66 μJy/beam
J1615 at 125.5 AU: sigma = 189.42 μJy/beam
J1842 at 63.2 AU: sigma = 118.33 μJy/beam
J1852 at 30.9 AU: sigma = 48.87 μJy/beam
LkCa_15 at 14.6 AU: sigma = 39.08 μJy/beam
LkCa_15 at 86.3 AU: sigma = 119.04 μJy/beam
MWC_758 at 30.1 AU: sigma = 1807.4

In [7]:
# add back as an extra column to gaps_with_sigma.csv 
gap_df['sigma_uJy_per_beam'] = [
    all_disks[disk_name].get_sigma_at_radius(radius_au=radius_au, robust_val="2.0", use_full_fov=False)
    if disk_name in all_disks else np.nan
    for disk_name, radius_au in zip(gap_df['disk_name'], gap_df['radius_au'])
]
gap_df.to_csv(r'd:\CPD_MPIA\Median_SNR\disk_object_files\gaps_with_sigma.csv', index=False)